In [20]:
import pandas as pd
import re
import torch # If using a GPU
from transformers import AutoTokenizer, AutoModelForTokenClassification
from transformers import pipeline

In [21]:
dbert_tokenizer = AutoTokenizer.from_pretrained("dslim/distilbert-NER")
dbert_model = AutoModelForTokenClassification.from_pretrained("dslim/distilbert-NER")
dbert_nlp = pipeline("ner", model=dbert_model, tokenizer=dbert_tokenizer, aggregation_strategy="max")

ConnectionError: (MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /api/models/dslim/distilbert-NER/tree/main/additional_chat_templates?recursive=False&expand=False (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x0000016D37D13680>: Failed to resolve \'huggingface.co\' ([Errno 11001] getaddrinfo failed)"))'), '(Request ID: 2a524a18-0f01-4491-b83e-64e831804521)')

In [ ]:
uncased_tokenizer = AutoTokenizer.from_pretrained("issifuamajeed/distilbert-base-uncased-finetuned-ner")
uncased_model = AutoModelForTokenClassification.from_pretrained("issifuamajeed/distilbert-base-uncased-finetuned-ner")
uncased_nlp = pipeline("ner", model=uncased_model, tokenizer=uncased_tokenizer, aggregation_strategy="max")

In [4]:
# Get Kaggle dataset
reviews_df = pd.read_csv('datasets/food_reviews_with_product_names.csv')
reviews_df = reviews_df.set_index('Id')
reviews_df

,ProductId,Score,Summary,Text,ItemName
Id,,,,,
5,B006K2ZZ7K,5,Great taffy,Great taffy at a great price. There was a wid...,"Salt Water Taffy - Assorted, 5 lbs"
6,B006K2ZZ7K,4,Nice Taffy,I got a wild hair for taffy and ordered this f...,"Salt Water Taffy - Assorted, 5 lbs"
7,B006K2ZZ7K,5,Great! Just as good as the expensive brands!,This saltwater taffy had great flavors and was...,"Salt Water Taffy - Assorted, 5 lbs"
8,B006K2ZZ7K,5,"Wonderful, tasty taffy",This taffy is so good. It is very soft and ch...,"Salt Water Taffy - Assorted, 5 lbs"
14,B001GVISJM,4,fresh and greasy!,good flavor! these came securely packed... the...,"TWIZZLERS Twists, Strawberry Flavored Licorice..."
...,...,...,...,...,...
568447,B001EO7N10,2,Mixed wrong,I had ordered some of these a few months back ...,"Taste Specialty Foods, Five Spice Powder, 16-O..."
568448,B001EO7N10,5,"If its all natural, this is like panacea of Sp...","Hoping there is no MSG in this, this tastes ex...","Taste Specialty Foods, Five Spice Powder, 16-O..."
568449,B001EO7N10,5,Very large ground spice jars.,My only complaint is that there's so much of i...,"Taste Specialty Foods, Five Spice Powder, 16-O..."


In [5]:
reviews_4star_df = reviews_df[reviews_df['Score'] == 4]
reviews_4star_df

,ProductId,Score,Summary,Text,ItemName
Id,,,,,
6,B006K2ZZ7K,4,Nice Taffy,I got a wild hair for taffy and ordered this f...,"Salt Water Taffy - Assorted, 5 lbs"
14,B001GVISJM,4,fresh and greasy!,good flavor! these came securely packed... the...,"TWIZZLERS Twists, Strawberry Flavored Licorice..."
28,B001GVISJM,4,Great Bargain for the Price,I was so glad Amazon carried these batteries. ...,"TWIZZLERS Twists, Strawberry Flavored Licorice..."
33,B001EO5QW8,4,Best of the Instant Oatmeals,McCann's Instant Oatmeal is great if you must ...,"McCANN'S Instant Irish Oatmeal, Variety Pack o..."
34,B001EO5QW8,4,Good Instant,This is a good instant oatmeal from the best o...,"McCANN'S Instant Irish Oatmeal, Variety Pack o..."
...,...,...,...,...,...
568318,B0013Z0PTW,4,Good Tasting Granola Bars,Most health bars taste awful. Fiber One Grano...,"Fiber One Chewy Bars, Oats and Caramel, 5 - 1...."
568379,B000H28ABW,4,Good and natural,Grab some flavor and save yourself some chemic...,Wick Fowler's Famous Taco Seasoning 1.25 oz. P...
568407,B0039KE8Y2,4,great taste,This apple butter has a great taste but the pr...,"Homestyle Apple Butter, 4.5 oz"


In [7]:
# Uses a Huggingface NER pipeline to find matches
def pipeline_entity_ranges(input_text, pipeline, avoid_list=[]):
    # Returns: list of dictionaries ([{type, value, begin, end}])
    entity_ranges = []
    avoid_list = [al.lower() for al in avoid_list]
    ner_results = pipeline(input_text)
    for entry in ner_results:
        if entry['word'].lower() not in avoid_list:
            entity_ranges.append({'type': entry['entity_group'],
                                  'value': entry['word'],
                                  'begin': entry['start'],
                                  'end': entry['end']}
                                )
    return(entity_ranges)

In [8]:
# Uses regex to find additional entity matches

def regex_entity_ranges(input_text, regex_list, avoid_list=[]):
    # regex_list = [[type, expression]],
    #    e.g., ['PHONE', r'[\+]?[(]?[0-9]{3}[)]?[-\s\.]?[0-9]{3}[-\s\.]?[0-9]{4,6}']
    
    matches = []
    avoid_list = [al.lower() for al in avoid_list]
    # The list determines order. If any matches overlap, this function keeps the first match found and discards the newer one
    for match_entry in regex_list:
        (etype, exp) = match_entry
        for re_match in re.finditer(exp, input_text):
            overlap = False
            for existing_match in matches:
                if re_match.start() <= existing_match['end'] and existing_match['begin'] <= re_match.end():
                    overlap = True
                    break

            if not overlap and re_match.group().lower() not in avoid_list:
                matches.append({'type': etype,
                                'value': re_match.group(),
                                'begin': re_match.start(), 
                                'end': re_match.end()}
                              )
    return matches

In [9]:
# Combines output from the above two functions
def find_all_entities(text, pipelines=[], regex_patterns=[], avoid_list=[] ):
    intermediate_results = []
    if type(pipelines) != type([]):
        pipelines = [pipelines]
    for pipeline in pipelines:
        intermediate_results += pipeline_entity_ranges(text, pipeline, avoid_list)
        
    intermediate_results += regex_entity_ranges(text, regex_patterns, avoid_list)

    # Deduplicate / de-overlap
    #### TODO ####
    # Right now, this block will drop anything that overlaps previously-found entities.
    # For instance if Model 1 finds "Newcastle" and Model 2 finds "Newcastle upon Tyne", the better match gets dropped
    # We could modify the existing match to least(beginning) - greatest(ending), but what if the entity types are different?
    # In this case we may need to create a new "entity type" of "Unknown" or similar?
    final_results = []
    for ir in intermediate_results:
        add_to_final = True
        for fr in final_results:
            if ir['begin'] <= fr['end'] and fr['begin'] <= ir['end']:
                add_to_final = False
        if add_to_final:
            final_results.append(ir)

    return(final_results)

In [10]:
# Given a list of matches, redacts text
def redact_text(text, found_entity_list=[]):
    text_chunks = []
    last_end_pos = 0
    # The logic below needs the entity list in left-to-right order to work.
    sorted_entity_list = sorted(found_entity_list, key=lambda x: x['begin'])
    
    for entity in sorted_entity_list:
        (etype, ebegin, eend) = (entity['type'], entity['begin'], entity['end'])
        text_chunks.append(text[last_end_pos:ebegin])
        # Standardize type redactions for comparison
        redact_dict = {'PER': 'PERSON'}
        text_chunks.append(f"""[{redact_dict.get(etype) or etype}]""")
        last_end_pos = eend
    text_chunks.append(text[last_end_pos:])
    return ''.join(text_chunks)

In [10]:
# Many product names have people/place names in them. Add the whole product name and two-word chunks to the ignore list.
# Also, these are Amazon reviews, so we'll add "Amazon"
def product_name_to_ignore_list(product_name):
    ignore_list = str(product_name).split(' ')
    for i in range(len(ignore_list) - 1):
        ignore_list.append(' '.join(ignore_list[i:i+2]))
    ignore_list.append('Amazon')
    return ignore_list

In [11]:
regex_list = [['PHONE',r'[\+]?[(]?[0-9]{3}[)]?[-\s\.]?[0-9]{3}[-\s\.]?[0-9]{4,6}'],
              ['NUMBER',r'[0-9]{3,}']
             ]

In [13]:
pd.DataFrame(
find_all_entities('My phone number is (315) 555-1212 and I live at 9980 Timberwood Circle.',[dbert_nlp],regex_list,[])
)

,type,value,begin,end
0,LOC,Timberwood Circle,53,70
1,PHONE,(315) 555-1212,19,33
2,NUMBER,9980,48,52


In [14]:
test_1 = 'Robert lives in Fishers, Indiana. His wife Linda works at Olive Garden.'
pd.DataFrame(find_all_entities(test_1, dbert_nlp))

,type,value,begin,end
0,PER,Robert,0,6
1,LOC,Fishers,16,23
2,LOC,Indiana,25,32
3,PER,Linda,43,48
4,LOC,Olive Garden,58,70


In [15]:
test_1_res = find_all_entities(test_1, dbert_nlp)
print(redact_text(test_1, test_1_res))

[PERSON] lives in [LOC], [LOC]. His wife [PERSON] works at [LOC].


In [16]:
# Try it with more nonsense words not in a corpus
test_2 = 'Juspana lives in Polto, Unamia. Her wife Reke works at Norberop.'
pd.DataFrame(find_all_entities(test_2, dbert_nlp))

,type,value,begin,end
0,PER,Juspana,0,7
1,LOC,Polto,17,22
2,LOC,Unamia,24,30
3,PER,Reke,41,45
4,ORG,Norberop,55,63


In [17]:
test_2_res = find_all_entities(test_2, dbert_nlp)
print(redact_text(test_2, test_2_res))

[PERSON] lives in [LOC], [LOC]. Her wife [PERSON] works at [ORG].


In [18]:
# This just combines the find and redact functions
def process_text(text):
    entities = find_all_entities(text, [dbert_nlp, uncased_nlp])
    return redact_text(text, entities)

### TRY THIS FOR YOURSELF. GIVE ME A SENTENCE.

In [22]:
# This one is hard mode. Mixes names that may also be English words. And no capitalization.
sentence = "Who's on first? Exactly!"
print(process_text(sentence))

Who's on first? Exactly!


### Do this for the Google Reviews dataset. Store results in CSV.
#### As we go, we'll print the first few examples of both no redaction.

In [19]:
# BRING IT ALL TOGETHER!
def process_review(text, product_name):
    ignore_list = product_name_to_ignore_list(product_name)
    entities = find_all_entities(text, [dbert_nlp, uncased_nlp], regex_patterns=regex_list, avoid_list=ignore_list)
    redacted = redact_text(text, entities)
    return redacted
    
def process_review_row(row):
    return process_review(row['Text'], row['ItemName'])

In [22]:
# This still doesn't work too well with possessives. Need to look at this more.
process_review('I WAS VISITING MY FRIEND NATE THE OTHER MORNING AND WE HAD MCCANN''S OATMEAL', 'MCCANN''S')

'I WAS VISITING MY FRIEND [PERSON] THE OTHER MORNING AND WE HAD MCCANNS OATMEAL'

In [23]:
# This can take a little while to run on a laptop. It's going through about 25K reviews.
# It would go faster on a GPU with some re-tooling, but this is just a demo.
reviews_4star_df['RedactedReview'] = reviews_4star_df.apply(process_review_row, axis=1)

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
C:\Users\BrittonGray\AppData\Local\Temp\ipykernel_14012\479608492.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reviews_4star_df['RedactedReview'] = reviews_4star_df.apply(process_review_row, axis=1)


In [24]:
reviews_4star_df = reviews_4star_df.drop(columns=['ProductId','Score','Summary'])
reviews_4star_df

,Text,ItemName,RedactedReview
Id,,,
6,I got a wild hair for taffy and ordered this f...,"Salt Water Taffy - Assorted, 5 lbs",I got a wild hair for taffy and ordered this f...
14,good flavor! these came securely packed... the...,"TWIZZLERS Twists, Strawberry Flavored Licorice...",good flavor! these came securely packed... the...
28,I was so glad Amazon carried these batteries. ...,"TWIZZLERS Twists, Strawberry Flavored Licorice...",I was so glad Amazon carried these batteries. ...
33,McCann's Instant Oatmeal is great if you must ...,"McCANN'S Instant Irish Oatmeal, Variety Pack o...",[ORG]'s Instant Oatmeal is great if you must h...
34,This is a good instant oatmeal from the best o...,"McCANN'S Instant Irish Oatmeal, Variety Pack o...",This is a good instant oatmeal from the best o...
...,...,...,...
568318,Most health bars taste awful. Fiber One Grano...,"Fiber One Chewy Bars, Oats and Caramel, 5 - 1....",Most health bars taste awful. Fiber One [ORG]...
568379,Grab some flavor and save yourself some chemic...,Wick Fowler's Famous Taco Seasoning 1.25 oz. P...,Grab some flavor and save yourself some chemic...
568407,This apple butter has a great taste but the pr...,"Homestyle Apple Butter, 4.5 oz",This apple butter has a great taste but the pr...


In [25]:
rlist = reviews_4star_df[reviews_4star_df['RedactedReview'].str.contains(r'\[PERSON\]')]['RedactedReview'].tolist()
print('\n\n'.join(rlist[0::20]))

I WAS VISITING MY FRIEND [PERSON] THE OTHER MORNING FOR COFFEE , HE CAME OUT OF [MISC] STORAGE ROOM WITH ( A PACKET OF [PERSON] INSTANT IRISH OATMEAL .) HE SUGGESTED THAT I TRY IT FOR MY OWN USE ,IN MY STASH . SOMETIMES [PERSON] DOSE NOT GIVE YOU A CHANCE TO SAY NO , SO I ENDED UP TRYING THE APPLE AND [LOC] . FOUND IT TO BE VERY TASTEFULL WHEN MADE WITH WATER OR POWDERED MILK . IT GOES GOOD WITH O.J. AND COFFEE AND A SLICE OF TOAST AND YOUR READY TO TAKE ON THE WORLD...OR THE DAY AT LEAST..  [PERSON]...

These [PERSON]'s Signature Beyond Gourmet "Bacon" jelly beans do indeed taste like a very sweet and smokey piece of bacon. This is not flavor that I'm going to snack on, but would make a great gift for the many "bacon fanatics" out there. My wife commented that with a little vinegar flavor added in, they would taste like [MISC] [MISC] Dressing.<br /><br />This flavor is from [PERSON], the creator of the famous "[MISC]" jelly beans, so I was thrilled to be offered this in <a href="http:

In [26]:
reviews_4star_df.to_csv('datasets/outputs/food_reviews_huggingface.csv')